In this notebook, we enter the modeling phase of our predictive maintenance project. We will experiment with different models, perform hyperparameter tuning, and finally evaluate model performance on the NASA test set (`test_FD001.txt` and `RUL_FD001.txt`).

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Loading Features

In [2]:
# Load the features dataframe saved in notebook 2
df = pd.read_parquet('../data/processed/fd001_features.parquet.parquet')

In [3]:
df.columns

Index(['engine_number', 'time_cycle', 'setting_1', 'setting_2', 'setting_3',
       'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8',
       'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14',
       'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21', 'RUL',
       'sensor_2_roll_mean_17', 'sensor_3_roll_mean_17',
       'sensor_4_roll_mean_17', 'sensor_6_roll_mean_17',
       'sensor_7_roll_mean_17', 'sensor_8_roll_mean_17',
       'sensor_9_roll_mean_17', 'sensor_11_roll_mean_17',
       'sensor_12_roll_mean_17', 'sensor_13_roll_mean_17',
       'sensor_14_roll_mean_17', 'sensor_15_roll_mean_17',
       'sensor_17_roll_mean_17', 'sensor_20_roll_mean_17',
       'sensor_21_roll_mean_17', 'sensor_2_roll_std_10',
       'sensor_3_roll_std_10', 'sensor_4_roll_std_10', 'sensor_6_roll_std_10',
       'sensor_7_roll_std_10', 'sensor_8_roll_std_10', 'sensor_9_roll_std_10',
       'sensor_11_roll_std_10', 'sensor_12_roll_std_10',
       'sensor_13_roll_std_10

# Train/Validation Split

We perform an engine-based train/validation split to avoid data leakage. Since each engine represents an independent time series, splitting at row level would leak temporal and engine-specific patterns.

Instead, we split by engine IDs so that all cycles of a given engine are contained in either the training or validation set.

We use a 70/30 split (instead of 80/20) to obtain a slightly larger validation set, which leads to more stable evaluation metrics and a more reliable estimate of generalization across different degradation trajectories.

The exact ratio is not critical; the key principle is strict separation at engine level.

After the split, we remove `engine_number` and `time_cycle` from the feature set, as they are only required for splitting and time-based processing, not for model training.

In [6]:
# Get all engines
engines = df["engine_number"].unique()
engines

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100])

In [7]:
# Engine-level split
train_engines, val_engines = train_test_split(
    engines,
    test_size=0.3,
    random_state=42
)

In [ ]:
# Filter data by engines
train_df = df[df["engine_number"].isin(train_engines)].copy()
val_df = df[df["engine_number"].isin(val_engines)].copy()

In [10]:
train_df["engine_number"].unique()

array([  2,   3,   4,   6,   7,   8,   9,  12,  14,  15,  17,  18,  20,
        21,  22,  24,  25,  26,  28,  29,  30,  33,  35,  36,  37,  38,
        39,  42,  44,  47,  48,  49,  50,  51,  52,  53,  55,  57,  58,
        59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,  72,  75,
        76,  79,  80,  82,  83,  85,  86,  87,  88,  90,  92,  93,  94,
        95,  96,  98,  99, 100])

# Defining X and y

In this step, we separate the dataset into features (X) and the target variable (y). This distinction is required to prepare the data for model training, where X represents the input variables and y corresponds to the Remaining Useful Life (RUL).

Additionally, we remove the `engine_number` column from the feature set, as it is only an identifier and does not contain predictive information.

The `time_cycle` column is initially retained, as it may contain meaningful information about the operating duration of each engine and can serve as a relevant predictor for degradation modeling. Its influence will be evaluated during model training and feature importance analysis. In a second experiment, we will also test a variant where `time_cycle` is removed to assess its impact on model performance.

In [38]:
X_train = train_df.drop(["RUL", "engine_number"], axis=1)

In [13]:
y_train = train_df["RUL"]

In [39]:
X_val = val_df.drop(["RUL", "engine_number"], axis=1)

In [15]:
y_val = val_df["RUL"]

# Baseline Model

We start with a simple baseline model to serve as a reference for comparison. We train a basic linear regression model (without regularization) on the training set and evaluate its performance on the validation set.

For this purpose, we standardize the features for the baseline model. This is done for several reasons:

1. **More stable coefficients**  
   Standardization makes the intercept more interpretable and ensures that model weights become comparable across features.

2. **Improved numerical stability**  
   This is especially important in the presence of many correlated features, as it helps prevent feature dominance caused by differing value scales.

3. **Better interpretability**  
   After scaling, the importance of features becomes more comparable, allowing a clearer understanding of which variables contribute most to the prediction.

Furthermore, we train the baseline model twice: once including the `time_cycle` feature and once without it. This allows us to evaluate the impact of `time_cycle` on both model performance and learned parameters.

In [40]:
scaler = StandardScaler()

In [41]:
X_train_standardized = scaler.fit_transform(X_train)

In [42]:
X_val_standardized = scaler.transform(X_val)

In [43]:
baseline_model = LinearRegression()

In [44]:
baseline_model.fit(X_train_standardized, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](79,)","[-15.85, -0.42, 0.66,..., 2.72, -0.13, 2.23]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,101.3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,79
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,78
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](79,)","[561. ,303.16,172.08,..., 9.92, 5.01, 0. ]"


In [47]:
# Print the optimal weight (slope) and bias (intercept)
print("Coefficients (Slope):", baseline_model.coef_)
print("Intercept (Bias):", baseline_model.intercept_)

# Predict and score using test data
y_pred = baseline_model.predict(X_val_standardized)
print("Mean Squared Error:", mean_squared_error(y_val, y_pred))
print("R-squared Score:", r2_score(y_val, y_pred))

Coefficients (Slope): [-1.58465159e+01 -4.16691931e-01  6.61696436e-01  1.01163522e-12
 -1.24944804e-02 -1.02567284e-01  8.86392929e-02  1.10852506e-01
  5.27928473e-01  3.60287602e-01  2.31067912e-01 -4.13866178e-01
  5.62606376e-02  4.26021161e-01  2.81692028e-01  1.93369098e-01
 -2.82665394e-01  8.72583133e-02  1.42775038e-02 -1.00893265e+01
 -6.78147860e-01 -7.19865268e+00 -1.18292195e+00  3.93989325e+00
  3.88164173e+00  2.87391757e+01 -1.95388348e+01 -8.18847284e+00
  2.27105411e+01 -1.95742025e+01 -1.04251187e+01 -2.57129282e+00
 -3.49901259e+00  1.08962782e+01  3.73147340e-02  9.21417905e-02
 -9.85438767e-01 -7.20394906e-01  7.55490116e-01 -1.27579609e+00
 -1.03044960e+00  3.73331588e-01 -5.91096299e-02  1.87013762e-01
  1.03846427e-01 -9.93511396e-01  6.32042232e-01 -8.19340831e-01
  1.06670279e+00  1.12904244e-01 -2.36003357e+00 -1.68797520e-01
 -1.37943199e+00 -1.37132314e-01 -3.07422995e+00  6.71208907e-02
 -9.59667201e-02  2.70667280e-02  3.15118513e+00  2.01762013e-01
 -3

An R² score of approximately 0.73 is achieved, meaning that around 73% of the variance in the Remaining Useful Life (RUL) is explained by the baseline model.

For the FD001 dataset, which contains noisy and highly dynamic sensor signals, this represents a strong and realistic performance for a linear baseline model. It also indicates that the engineered features already contain meaningful predictive signal.

The mean squared error of approximately 998 may initially appear high. However, RUL values typically range from 0 to 200+, and the squared error metric disproportionately penalizes larger deviations. For a baseline model, this result is therefore acceptable, and the key aspect is the comparison to more advanced models.

The learned coefficients show varying influence strengths across features. Negative coefficients indicate that increasing feature values lead to a decrease in predicted RUL, which is consistent with degradation behavior. Positive coefficients indicate the opposite effect.

The wide range of coefficient magnitudes (approximately -20 to +28) suggests strong feature correlation and redundancy within the engineered feature set, particularly due to overlapping rolling features. As a result, the linear model distributes weights unevenly across correlated inputs.

Individual coefficients cannot be interpreted in isolation but must be understood in the context of the full feature space.

The intercept (~101.30) represents the baseline prediction of the model when all standardized features are equal to zero, i.e., when the engine is in an average operating state.

In practical terms, this means the model estimates a remaining useful life of approximately 101 cycles for a typical engine condition. This is consistent with the dataset characteristics, where engines operate on average around this range before reaching failure.

In the following, we train a second variant of the baseline model without the `time_cycle` feature. We then briefly evaluate its performance and compare it to the original baseline model to assess the impact of `time_cycle` on predictive performance and learned parameters.

In [48]:
X_train_without_time_cycle = X_train.drop("time_cycle", axis=1)
X_val_without_time_cycle = X_val.drop("time_cycle", axis=1)

In [53]:
scaler_2 = StandardScaler()

In [54]:
X_train_without_time_cycle_standardized = scaler_2.fit_transform(X_train_without_time_cycle)

In [55]:
X_val_without_time_cycle_standardized = scaler_2.transform(X_val_without_time_cycle)

In [56]:
baseline_model_without_time_cycle = LinearRegression()

In [57]:
baseline_model_without_time_cycle.fit(X_train_without_time_cycle_standardized, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](78,)","[-0.29, 0.29,-0. ,..., 2.95,-0.16, 2.16]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,101.3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,78
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,77
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](78,)","[555.24,302.65,172.07,..., 9.92, 5.01, 0. ]"


In [58]:
# Print the optimal weight (slope) and bias (intercept)
print("Coefficients (Slope):", baseline_model_without_time_cycle.coef_)
print("Intercept (Bias):", baseline_model_without_time_cycle.intercept_)

# Predict and score using test data
y_pred = baseline_model_without_time_cycle.predict(X_val_without_time_cycle_standardized)
print("Mean Squared Error:", mean_squared_error(y_val, y_pred))
print("R-squared Score:", r2_score(y_val, y_pred))

Coefficients (Slope): [-2.87293920e-01  2.89528073e-01 -2.13162821e-14  2.83174571e-02
 -2.47967309e-02  2.65300643e-01  1.57480970e-01  5.07180170e-01
  3.51399305e-01 -9.49506788e-03 -4.00522926e-01 -7.29523026e-02
  5.20965496e-01  4.89332486e-01  2.54878547e-01 -3.41609658e-01
  1.60928176e-02  5.91803432e-02 -8.63871233e+00 -2.96658173e+00
 -1.20573192e+01 -1.44530262e+00  4.96558557e+00  9.97399122e+00
  1.97442358e+01 -2.19268459e+01 -4.15864928e+00  2.63434862e+01
 -1.50533373e+01 -7.09279828e+00 -4.96570806e+00  1.04634646e+00
  1.35718427e+01 -6.05174883e-01  3.06947684e-02 -1.30967763e+00
 -1.15163941e+00  1.11410902e+00 -1.37227326e+00  4.09799728e-01
  5.00221099e-01 -4.36871299e-01 -9.56002169e-02  1.11413155e+00
 -8.45166460e-01  5.31785056e-01 -9.85777216e-01  1.25806790e+00
  1.72040758e-01 -2.49726824e+00 -2.24722640e-01 -1.45667211e+00
 -1.29140182e-01 -3.35435247e+00  7.00021218e-02 -3.04392605e-01
  9.49970346e-03  3.44787341e+00  2.27115787e-01 -3.38435358e+00
 -7

We observe that the model without the `time_cycle` feature performs slightly worse, achieving an R² score of around 0.67 and a mean squared error of approximately 1205.

This indicates that the `time_cycle` feature does contribute to explaining a small portion of the variance in the target variable. However, the improvement compared to the full model is relatively limited, suggesting that most of the predictive power is already captured by the engineered sensor-based features.

Overall, an R² of 0.67 is still acceptable for a linear baseline model in this setting, even though it is lower than the variant including `time_cycle`.